<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> AI Lab Education</h1>

# 🦓 Fine-Tune and Deploy Your Own Model: GRPO on Zebra Puzzles

In notebook 03 the agents debated two ways to post-train a small model:
reinforcement learning on verifiable rewards, or distillation from a larger
model. This notebook runs the first technique end to end — and this time the
model you serve at the end is **one you trained yourself**.

One thing to be clear about from the start, because the word is used loosely
everywhere: this notebook does **not** teach a model to reason. It makes a model
much more accurate at one bounded task — turning sentences into formal structure.
The reasoning is done by a constraint solver, which was already perfect at it.
That is a smaller claim than "reasoning", and a far more useful one, because the
result is measurable and the failure modes are inspectable.

Notebooks 00–03 consumed models the platform provided. This one produces one,
and takes it the whole way round: base model from the cluster's mirror, training
on your GPU, registration in Thinkube Experiments, serving on vLLM, and a call
through the LLM
Gateway that any application could make.

## 📌 Before you start

Run this from the top with a fresh kernel — the configuration in section 2 is
read by every later section, and a stale value from a previous run is the
easiest way to get a result that looks fine and is not.

If you watch the training cell, keep the browser tab open: its output belongs to
the session that started it. Section 10b writes the whole history to Thinkube
Experiments once training ends, so the record survives a lost tab either way.

## 🧩 The task, and the twist

A zebra puzzle: five houses, each with a colour, a nationality, a drink, a pet
and a cigarette brand, and fifteen clues like *"the Englishman lives in the
yellow house"* and *"the dog is next door to the fox"*. Fill in the grid.

The twist is what we ask the model to do with it. **The model never solves the
puzzle.** It translates each clue into a formal rule, and a constraint solver
deduces the grid from those rules:

> *"The Englishman lives in the yellow house."*
> → `{"type": "same", "a": ["nationality", "Englishman"], "b": ["color", "yellow"]}`

The model formalizes; the machine deduces. That division of labour is the point:
it is how knowledge graphs get built from documents, how questions become SQL,
how contracts become checkable records. And because a solver can check the
result, the training loop needs no labelled answers at all.

## 🎯 What You'll Learn

**The ML technique**
- ✅ **Reinforcement learning without labels** — GRPO, where a model improves by
  comparing its own attempts against each other
- ✅ **Verifiable rewards** — let a constraint solver grade the output, so no
  human writes answer keys and no reward model is needed
- ✅ **Formalize-then-solve** — the pattern behind knowledge graphs, text-to-query
  and document extraction
- ✅ **Honest evaluation** — a cached baseline, and a public benchmark the model
  never trained on

**The platform skills** (reusable in any project)
- ✅ **Model mirroring** — one cluster-wide copy instead of a download per node
- ✅ **The model registry in Thinkube Experiments** — runs vs registered models, and where the
  weights actually live
- ✅ **The model lifecycle** — `deployable → loading → available`, nodes, slots
- ✅ **The LLM Gateway** — serving your own fine-tune behind the same
  OpenAI-compatible endpoint as every stock model

## 🏗️ What We're Building

```mermaid
graph LR
    A[🤗 Hugging Face] -->|mirror, once| B[(📦 Thinkube Experiments + JuiceFS<br/>every node reads it)]
    B -->|load base model| C[🔧 GRPO Training<br/>Unsloth + TRL + LoRA]
    D[🧩 Puzzle generator<br/>+ CSP solver] -->|verifiable reward| C
    C -->|merge + register| B
    B -->|load_model| E[⚡ vLLM backend]
    E --> F[🔤 LLM Gateway<br/>OpenAI-compatible]
    F --> G[🐍 Your application]
```

## ⏱️ What it costs

**Full scale is the default: about 9.5 hours end to end on a single GPU**, of
which 6 h 54 min is the 250 training steps. That is what produced the results
this notebook reports — in-distribution accuracy 8% → 99%, and 28% → 67% on a
public benchmark the model never saw. Section 2 breaks the time down phase by
phase.

**If you would rather see it work first, set `FULL_RUN = False`.** Demo scale
runs 500 training puzzles, 50 in each evaluation set and 60 steps in **about an
hour**, and it produces a real, measurable improvement: in-distribution accuracy
roughly triples, from 6% to 20%. Every stage runs exactly as it does at full
scale — same reward, same evaluations, same registration and serving — so you
see the whole technique work, and the reward curve climb, inside a coffee break.

What an hour will not give you is a model worth deploying, or much benchmark
transfer: the large gains on unseen phrasing arrive with the longer run. Use
demo scale to watch the loop learn and to check your GPU, venv and gateway.
Use full scale when you want the model.

**Prerequisites**: the `fine-tuning` venv, one GPU with roughly 20 GB free, and
the platform services (Thinkube Experiments, LLM Gateway) that notebook 00
validated.

---
## 🎲 Why a Puzzle? What Makes a Good RL Training Task

Reinforcement learning does not need examples of good answers. It needs a
**grader** — something that can score an attempt without a human reading it. That
one requirement is what makes RL brilliant in some domains and useless in others,
and it is why the recent results in this field (DeepSeek-R1, Logic-RL) all come
from places where correctness can be checked by machine: mathematics, code,
formal logic.

Zebra puzzles are close to an ideal training ground, for six reasons worth
recognising when you look at your own problem:

| property | why it matters for RL |
|---|---|
| **Mechanically checkable** | A constraint solver decides correctness in milliseconds. No human labels, no reward model, nothing to argue with. |
| **Generated with ground truth** | The generator starts *from* a solution and invents clues for it, so every puzzle arrives with its answer key — training data is free and unlimited. |
| **Uniquely solvable** | The generator verifies exactly one arrangement satisfies the clues, so "correct" is never ambiguous. |
| **Partial credit** | Scoring 25 grid cells gives a gradient. A model that gets twelve rules right ranks above one that gets three, so it can climb. Pass/fail rewards on a task the model initially fails are silent. |
| **Uncontaminated** | Freshly generated puzzles cannot be in the base model's training data. A public dataset gives you no such guarantee, and a contaminated benchmark flatters every result you produce. |
| **Cheap and short** | GRPO generates eight answers per prompt per step. Short outputs and a fast grader mean many steps per hour; a task needing 4,000-token answers or a 30-second grader would make this notebook a week-long job. |

### ✅ Is *your* task a fit? Four questions

1. **Can a program decide whether an answer is right?** A test suite, a schema, a
   query that either returns the right rows, a checksum that either balances. If
   the only judge is a person, RL is the wrong tool — supervised fine-tuning on
   their judgements is the right one.
2. **Can you produce many examples with known answers?** Generate them, mine them
   from systems that already hold verified records, or reverse them from
   structured data you already own (as the generator here does).
3. **Does partial credit exist?** All-or-nothing rewards learn far more slowly.
   Look for something countable: fields correct, rows matched, tests passed.
4. **Are the outputs short enough to generate a lot of?** RL's cost scales with
   generations, not with dataset size.

If you answered yes four times, the loop in this notebook transfers to your
problem with the rule language and the solver swapped out.

### 🧭 And the honest caveat

The puzzle is a gym, not the destination. What the model actually acquires here
is the habit of turning a sentence into exact structure — and that habit is the
part that transfers. Nobody needs a model that fills in zebra grids; plenty of
people need one that turns a clinical note into coded fields, or a question into
a query that runs.

---
## Setup

**What**: Confirm the libraries this notebook needs are present.

**Why**: Three pieces do the work — **Unsloth** (memory-efficient LoRA fine-tuning),
**TRL** (the GRPO trainer itself), and **python-constraint** (the CSP solver that
turns rules into a grid). On thinkube's `fine-tuning` venv all three are already
installed, so this cell stays commented out.

**How**: Uncomment only if you are running somewhere else. Installing over a
working venv is the most common way to break a GPU stack — the versions here are
pinned together deliberately.

In [ ]:
# Uncomment to install:
# !pip install unsloth trl datasets python-constraint matplotlib

---
## 1. Imports and Environment

**What**: Load the libraries, and set three environment flags before Unsloth is
imported.

**Why**: Unsloth accelerates training by compiling the model's hot paths with
`torch.compile`, and that compilation is left **on** here — it is a large part
of why Unsloth is faster than a plain trainer. The two flags cleared below are
debugging switches that a previous session might have left set in the kernel's
environment; `CUDA_LAUNCH_BLOCKING` in particular makes every CUDA call
synchronous, which is invaluable when hunting a crash and ruinous for training
speed.

**How**: Flags that change Unsloth's behaviour must be set *before*
`import unsloth`, because the library reads them at import time and generates a
compiled trainer for your specific model on that basis. If you change one later,
you must restart the kernel **and** delete the `unsloth_compiled_cache/`
directory next to this notebook — otherwise the previously generated code is
reused and your change appears to do nothing.

**A warning worth understanding rather than fixing.** The container image pairs
its PyTorch with a torchvision one minor version behind what that PyTorch asks
for, and Unsloth says so on import. The obvious response — install a newer
torchvision — is the wrong one, and it is worth knowing why: torchvision
declares an *exact* PyTorch version, so pip would install a second PyTorch
inside your venv. That copy shadows the image's tuned build for everything
using the kernel, and every extension compiled against the original — FlashAttention
among them — stops loading. Nothing here uses torchvision, so the check is
switched off rather than satisfied.

`zebra_dataset` is the local module next to this notebook: it generates puzzles
and knows nothing about models.


In [ ]:
import json
import os
import random
import re
import time

os.environ.pop("UNSLOTH_RETURN_LOGITS", None)
os.environ.pop("CUDA_LAUNCH_BLOCKING", None)

# The base image ships a torchvision one minor version behind what this torch
# asks for, and Unsloth warns about it on import. Do NOT resolve that warning
# by installing a newer torchvision: torchvision pins an exact torch version,
# so pip would install a second PyTorch inside the venv and shadow the image's
# build, breaking every CUDA extension compiled against it. Nothing here uses
# torchvision, so the check is skipped instead.
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"

import torch
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset, load_dataset

import zebra_dataset

---
## 2. Configuration

**What**: Every number that shapes the run, in one cell.

**Why**: Two settings deserve explanation.

`FULL_RUN` picks the scale, and the honest advice is to leave it on.

**Full is an overnight job — budget about 9.5 hours.** Measured on one GB10
(DGX Spark) with Qwen3.5-4B, FlashAttention-2 and `torch.compile` enabled:

| phase | time |
|---|---|
| load base model from the mirror | ~1 min |
| generate 2,200 puzzles | ~3 min |
| baseline evaluation (2 × 200 puzzles) | ~75 min |
| **GRPO training, 250 steps** | **6 h 54 min** |
| post-training evaluation (2 × 200 puzzles) | ~31 min |
| merge, upload 8.7 GB, register | ~41 min |
| load on vLLM and call the gateway | ~5 min |
| **total** | **~9.5 hours** |

Two things shorten a re-run: the baseline is cached to disk and keyed by scale,
so the 75 minutes are paid once, and generation is the only CPU-bound step.

**Demo scale — `FULL_RUN = False` — shows you the technique working in about an
hour.** It runs 500 training puzzles, 50 in each evaluation set and 60 GRPO
steps. Every stage is identical to the full run: same reward, same evaluations,
same registration and serving. You watch the reward curve climb and you get a
real improvement — in-distribution accuracy roughly triples, 6% to 20%.

What an hour does not buy is a model worth deploying, or much transfer: at demo
scale the public benchmark barely moves, against 28% → 67% at full scale. The
gains on phrasing the model has never seen are what the longer run is for, and
they are the result worth quoting.

So: demo scale to see it learn and to check your GPU, venv and gateway end to
end. Full scale when you want the model.

`KL_BETA = 0.0` turns off the KL penalty. GRPO can hold the trained model near
its starting point by comparing against a frozen reference copy; setting beta to
zero removes both the penalty and the need to keep that second copy in memory.
That is TRL's own default, and for a short run on a narrow task the drift it
guards against is not the risk worth paying for.

The `EVAL_*` sampling values are not invented here — they are the settings the
Qwen3.5 model card publishes for non-thinking use, which is how the model is
meant to be run when you want a direct answer rather than visible deliberation.

**How**: Change `FULL_RUN` and re-run from this cell down. The two evaluation
sets keep separate seeds (`TRAIN_SEED`, `EVAL_SEED`) so no training puzzle can
appear in a test.

In [ ]:
MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LEN = 3072
LORA_RANK = 16
LORA_ALPHA = 16

TRAIN_SEED = 1
EVAL_SEED = 99
SEED = 3407

NUM_GENERATIONS = 8
LEARNING_RATE = 1e-5
GRADIENT_ACCUMULATION = 4
# 0 keeps TRL's default (no KL penalty, no reference model)
KL_BETA = 0.0
MAX_COMPLETION_LENGTH = 800

# Qwen3.5 model-card sampling for non-thinking evaluation runs
EVAL_TEMPERATURE = 0.7
EVAL_TOP_P = 0.8
EVAL_TOP_K = 20

OUTPUT_DIR = "./zebra_rules_qwen35_4b"

# Full scale by default: ~9.5 hours end to end, and the scale that produces a
# model worth serving. Set False for a ~1 hour run that still shows the reward
# curve climbing and in-distribution accuracy roughly tripling.
FULL_RUN = True

if FULL_RUN:
    NUM_TRAIN_PUZZLES = 2000
    NUM_EVAL_PUZZLES = 200
    NUM_BENCH_PUZZLES = 200
    MAX_STEPS = 250
else:
    NUM_TRAIN_PUZZLES = 500
    NUM_EVAL_PUZZLES = 50
    NUM_BENCH_PUZZLES = 50
    MAX_STEPS = 60

print(f"Model: {MODEL_NAME}")
print(f"Scale: {'FULL' if FULL_RUN else 'DEMO'}")
print(f"Train: {NUM_TRAIN_PUZZLES}, Eval: {NUM_EVAL_PUZZLES}, Bench: {NUM_BENCH_PUZZLES}")
print(f"LoRA rank: {LORA_RANK}, LR: {LEARNING_RATE}, Steps: {MAX_STEPS}")

---
## 3. Load the Base Model — and Meet Model Mirroring

**What**: Fetch Qwen3.5-4B and attach a LoRA adapter to it.

### 💡 Platform concept: mirroring

A model on Hugging Face is remote. A **mirrored** model has been copied once into
your cluster's Thinkube Experiments store, which lives on JuiceFS — a shared filesystem every
GPU node can read. That distinction decides how your day goes:

| | not mirrored | mirrored |
|---|---|---|
| where the weights live | Hugging Face | your cluster, on JuiceFS |
| who downloads them | every pod, every node, every time | nobody, after the first time |
| a 9 GB model on a slow uplink | ~20 minutes, repeatedly | seconds, from local storage |
| works offline | no | yes |

Mirroring is the same mechanism notebook 01 used to make chat models loadable —
here you are using it for a *training* base model, and later (section 14) you
will put your own fine-tune into the same store.

**How to mirror a model** — three equivalent routes:

```python
# 1. From a notebook or script, via thinkube-control's API
#    (also available as MCP tools, and in the UI under AI → Models)
from tk_llm import LLMClient
LLMClient().list_models()          # what the catalog offers, and its state

# 2. Submit the mirror job — returns immediately, Argo does the work
#    submit_model_mirror(model_id="unsloth/Qwen3.5-4B")
#
# 3. Watch it
#    get_mirror_status(workflow_id="model-dl-xxxxx")   # running → succeeded
```

A model must be in the **catalog** (`thinkube-metadata/models.json`) before it can
be mirrored — that file is what tells the platform a model exists, what backend
can serve it, and how big it is.

### 🔁 The pattern worth copying

`mirrored_model_path()` below asks Thinkube Experiments where the mirror is and returns a local
path, or `None`. Every notebook that loads a base model can use that shape:
**prefer the local mirror, fall back to the hub, and say which one you got.** It
costs ten lines and removes a per-node download from every future run.

### 💡 Why LoRA

Full fine-tuning updates all 4.5 billion parameters and needs optimiser state to
match. LoRA freezes the model and inserts small trainable matrices into the
attention and MLP projections: **21 million trainable parameters, 0.47% of the
model**. That is what fits this run on one GPU, and what makes the result a
~60 MB adapter instead of a second copy of the model.

### 💡 Attention backends, and a platform lesson about environments

This run uses `attn_implementation="flash_attention_2"`. FlashAttention-2 keeps
the attention computation in fast on-chip memory instead of materialising the
full attention matrix, which saves both time and memory.

The reason it is worth a section of its own is what it depends on. FA2 is a
**compiled CUDA extension**, built against a specific PyTorch. So is the
GatedDeltaNet kernel package, and so is `causal_conv1d`. A compiled extension
loaded against a different PyTorch than it was built for does not fail cleanly —
it fails strangely, with errors like
`Cannot access data pointer of Tensor that doesn't have storage`, which read as
bugs in your model code and are nothing of the sort.

**Where this bites on thinkube.** Your Jupyter environment is layered. The
container image ships PyTorch, CUDA, triton and flash-attn already compiled for
this cluster's GPUs — including GB10 / Grace Blackwell, whose `sm_121` kernels
are missing from stock public wheels. The venv you select as your kernel is built
with `--system-site-packages` so it **inherits** that PyTorch rather than
installing its own.

That inheritance is easy to break by accident. Install any package that declares
a hard `torch==` dependency and pip will quietly place a second PyTorch inside
your venv, where it shadows the tuned one for every process using that kernel.
Nothing announces this. What you notice, later, is compiled extensions failing in
ways that make no sense.

**The check worth knowing** — where does your PyTorch actually come from?

```python
import torch, os
print(torch.__version__, os.path.dirname(torch.__file__))
```

A path under `/usr/local/lib/python3.12/dist-packages` is the image's build, which
is what you want. A path inside `~/venvs/...` means your venv has its own copy and
is shadowing it. If you must add a torch-dependent package, install it with
`--no-deps` so pip cannot drag PyTorch along behind it.



In [ ]:
def mirrored_model_path(model_id):
    """Local JuiceFS path of the model's MLflow mirror, or None if absent.

    Mirrored models (thinkube-control -> Models -> mirror) are shared by all
    GPU nodes, so loading from the mirror avoids a per-node HuggingFace
    download. The registry is queried on the in-cluster address with the
    platform's MLflow token.
    """
    try:
        from pathlib import Path
        import thinkube_models as tkm
        from mlflow import MlflowClient

        token = tkm.get_mlflow_token()
        if not token:
            return None
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
        client = MlflowClient(tracking_uri="http://mlflow.mlflow.svc.cluster.local:5000")
        versions = client.search_model_versions(f"name='{model_id.replace('/', '-')}'")
        if not versions:
            return None
        latest = max(versions, key=lambda v: int(v.version))
        run = client.get_run(latest.run_id)
        cand = (Path.home() / "thinkube" / "mlflow" / "artifacts"
                / run.info.experiment_id / latest.run_id / "artifacts" / "model")
        return str(cand) if cand.exists() else None
    except Exception as e:
        print(f"Mirror lookup failed ({e}); falling back to HuggingFace")
        return None


model_source = mirrored_model_path(MODEL_NAME)
if model_source:
    print(f"Loading from local MLflow mirror: {model_source}")
else:
    model_source = MODEL_NAME
    print(f"No local mirror; downloading from HuggingFace: {MODEL_NAME}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_source,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    dtype=torch.bfloat16,
    fast_inference=False,
    # FlashAttention-2 requires that the flash-attn build match the torch it is
    # loaded against; both come from the base image here. The linear-attention
    # layers use their own fla kernels regardless of this setting.
    attn_implementation="flash_attention_2",
)

# LoRA on the full-attention projections and MLPs. The GatedDeltaNet
# linear-attention layers keep their base weights.
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,} total, {trainable:,} trainable ({100*trainable/total:.2f}%)")

---
## 4. Generate the Puzzles

**What**: Build training and evaluation puzzles with `zebra_dataset.py`.

**Why**: Reinforcement learning needs a grader more than it needs examples, and a
grader needs ground truth. Generating the puzzles gives us both for free: the
generator starts from a solution, invents clues that describe it, and then
**verifies with a CSP solver that exactly one arrangement satisfies them**. It
also minimises the clue set — remove any clue and the puzzle admits more than one
answer. So every puzzle is fair, unambiguous, and carries its own answer key,
without a human labelling anything.

Three themes (classic houses, a music school, a tapas bar) vary the vocabulary,
so the model cannot succeed by memorising that "zebra" belongs in the pet column.

**Why disjoint seeds**: training uses seed 1, evaluation seed 99, and the cell
asserts the two sets share no solution. An evaluation the model has already
trained on measures memory, not learning — and the assertion means you find that
out here rather than in the results.

**How**: Generation is CPU work and takes a couple of minutes at demo scale; the
solver rejects far more candidates than it keeps.

In [ ]:
def make_puzzles(num_puzzles, seed):
    """Generate puzzles with a fixed RNG."""
    rng = random.Random(seed)
    puzzles = []
    while len(puzzles) < num_puzzles:
        theme = rng.choice(zebra_dataset.THEMES)
        puzzle = zebra_dataset.generate(theme, 5, rng)
        if puzzle is None:
            continue
        puzzles.append(puzzle)
        if len(puzzles) % 200 == 0:
            print(f"  {len(puzzles)}/{num_puzzles}...")
    return puzzles

print("Generating training puzzles...")
train_puzzles = make_puzzles(NUM_TRAIN_PUZZLES, TRAIN_SEED)

print("Generating evaluation puzzles...")
eval_puzzles = make_puzzles(NUM_EVAL_PUZZLES, EVAL_SEED)

# Disjointness assertion
train_sigs = {json.dumps(p["solution"], sort_keys=True) for p in train_puzzles}
eval_sigs = {json.dumps(p["solution"], sort_keys=True) for p in eval_puzzles}
assert train_sigs.isdisjoint(eval_sigs), "train/eval overlap detected — check seeds"
print(f"Train/eval disjoint: confirmed ({len(train_sigs)} vs {len(eval_sigs)} unique)")

---
## 5. Load the Public Benchmark

**What**: Load ZebraLogic — 1,000 puzzles written by other people — and convert
them into the same shape our generator produces.

**Why**: Our own evaluation set proves the model learned *our* clue phrasing.
That is worth knowing and it is not the interesting question. ZebraLogic
describes the same kind of puzzle in different words — "The person who loves
fantasy books is the Norwegian" — so success here means the translation skill
generalises beyond the sentences it was trained on. Notebook 03 debated exactly
this benchmark; now it becomes the referee.

**Why this copy of it**: the dataset exists in two places. `allenai/ZebraLogicBench`
publishes the puzzles with their solutions **redacted** — every cell reads `___`,
because it is a leaderboard set and the answers are deliberately withheld.
`WildEval/ZebraLogic` is the same benchmark from the same authors with the
solutions intact, which is what a local grader needs.

**How**: The conversion also lifts the numbered clue sentences out of the prompt
text, because the clues are what the model must translate. Puzzles come in sizes
from 2×2 to 6×6; we keep the 5-house ones so the difficulty matches our
generator's.

In [ ]:
# WildEval/ZebraLogic is the ZebraLogicBench release with unredacted solutions
# (allenai/ZebraLogicBench hides them; the -private variant is gated)
zlogic = load_dataset("WildEval/ZebraLogic", "grid_mode", split="test")
print(f"ZebraLogicBench: {len(zlogic)} puzzles")
print(f"Schema: {zlogic.features}")
print(f"\nSample row keys: {list(zlogic[0].keys())}")
print(f"Sample size: {zlogic[0].get('size', 'N/A')}")

In [ ]:
def zlogic_to_internal(row):
    """Convert a ZebraLogic grid_mode row to our puzzle dict shape.

    Returns (puzzle_dict, original_prompt) or (None, None) if conversion fails.
    Rows look like: size "5*6" (houses*attributes), solution
    {"header": ["House", cat1, ...], "rows": [["1", val, ...], ...]}.
    """
    try:
        size_str = row.get("size", "")
        if "*" not in size_str:
            return None, None
        n_houses = int(size_str.split("*")[0])

        sol_raw = row.get("solution", {})
        if isinstance(sol_raw, str):
            sol_raw = json.loads(sol_raw)
        if not (isinstance(sol_raw, dict) and "header" in sol_raw and "rows" in sol_raw):
            return None, None

        header = sol_raw["header"]
        rows = sol_raw["rows"]
        if len(rows) != n_houses:
            return None, None

        # Order houses by the House column, then read one category per column
        house_col = next(i for i, cat in enumerate(header) if cat.lower() == "house")
        rows = sorted(rows, key=lambda r: int(r[house_col]))

        solution = {}
        categories = {}
        for col, cat in enumerate(header):
            if col == house_col:
                continue
            values = [r[col] for r in rows]
            solution[cat] = values
            categories[cat] = sorted(set(values))

        # The benchmark's own clue sentences are the transfer test: extract the
        # numbered lines so the model must translate unfamiliar phrasing
        original_prompt = row.get("puzzle", "")
        clues = re.findall(r"^\s*\d+\.\s+(.+?)\s*$", original_prompt, re.MULTILINE)
        if not clues:
            clues = [original_prompt]

        puzzle = {
            "N": n_houses,
            "categories": categories,
            "solution": solution,
            "clues": clues,
        }
        return puzzle, original_prompt

    except Exception:
        return None, None


# Convert and filter to 5-house puzzles
zlogic_puzzles = []
zlogic_all_sizes = {}

for row in zlogic:
    puzzle, orig_prompt = zlogic_to_internal(row)
    if puzzle is None:
        continue
    size = row.get("size", "?")
    zlogic_all_sizes.setdefault(size, []).append((puzzle, orig_prompt))
    if puzzle["N"] == 5:
        zlogic_puzzles.append(puzzle)

print(f"\nZebraLogic conversion:")
for size, items in sorted(zlogic_all_sizes.items()):
    print(f"  {size}: {len(items)} puzzles")
print(f"\n5-house puzzles for evaluation: {len(zlogic_puzzles)}")
print(f"Sample bench clue: {zlogic_puzzles[0]['clues'][0][:100]}")

---
## 6. The Rule Language and the Solver

**What**: Define the ten rule types the model may emit, the prompt that teaches
them, the solver that executes them, and the grader that scores the result.

**Why**: This is the heart of the notebook. The model's entire job is to turn a
sentence into one JSON object:

> *"The Englishman lives in the yellow house."*
> `{"type": "same", "a": ["nationality", "Englishman"], "b": ["color", "yellow"]}`

Ten rule types cover the clue vocabulary:

| rule | meaning |
|---|---|
| `at` / `not_at` | an attribute is (or is not) in a given house |
| `same` | two attributes share a house |
| `immediate_left` / `immediate_right` | directly beside, in that order |
| `next_to` | beside, either side |
| `left_of` / `right_of` | anywhere to that side |
| `one_between` / `two_between` | exactly one or two houses apart |

`solve_rules` turns those into a constraint problem — every attribute gets a
house number, values within a category are all different — and asks the solver
for an arrangement. **Malformed rules are skipped, never fatal.** The model is
producing this text, so it will invent fields, misname a value, or emit prose;
each of those simply costs accuracy. Contradictory rules make the puzzle
unsolvable, which scores zero. A perfect translation yields the one true grid.

**Why grade cells rather than right-or-wrong**: `grade_grid` returns the fraction
of the 25 grid cells that match. That gradient is what makes learning possible —
a translation with twelve rules right lands above one with three, and the model
can climb. A pass/fail reward on a task it fails at first would be silent.

**How**: The training rows carry the prompt plus the answer key and clue count,
so the reward function in section 8 can re-derive everything it needs.

In [ ]:
from constraint import Problem, AllDifferentConstraint

RULE_FORMS = """{"type":"at","item":["category","value"],"house":2}
{"type":"not_at","item":["category","value"],"house":2}
{"type":"same","a":["category","value"],"b":["category","value"]}
{"type":"immediate_left","a":["category","value"],"b":["category","value"]}
{"type":"immediate_right","a":["category","value"],"b":["category","value"]}
{"type":"next_to","a":["category","value"],"b":["category","value"]}
{"type":"left_of","a":["category","value"],"b":["category","value"]}
{"type":"right_of","a":["category","value"],"b":["category","value"]}
{"type":"one_between","a":["category","value"],"b":["category","value"]}
{"type":"two_between","a":["category","value"],"b":["category","value"]}"""


def translation_prompt(puzzle):
    cats_block = "\n".join(f"  {c}: {', '.join(vs)}" for c, vs in puzzle["categories"].items())
    clues_block = "\n".join(f"  {i}. {c}" for i, c in enumerate(puzzle["clues"], 1))
    return (
        "Translate each zebra-puzzle clue into one formal rule.\n\n"
        f"Categories and values:\n{cats_block}\n\n"
        f"Houses are numbered 1 to {puzzle['N']} from left to right.\n"
        "'immediate_left' means a is directly left of b; 'left_of' means a is anywhere left of b.\n"
        "'one_between' means exactly one house stands between a and b.\n\n"
        f"Rule forms (JSON, one per clue):\n{RULE_FORMS}\n\n"
        "Answer with ONLY a JSON array containing one rule per clue, in order. "
        "Copy category and value names exactly as given.\n\n"
        f"Clues:\n{clues_block}"
    )


def parse_rules(text):
    m = re.search(r"\[.*\]", text, re.DOTALL)
    if not m:
        return None
    try:
        rules = json.loads(m.group(0))
        return rules if isinstance(rules, list) else None
    except Exception:
        return None


_PAIR_CONSTRAINTS = {
    "same": lambda a, b: a == b,
    "immediate_left": lambda a, b: a == b - 1,
    "immediate_right": lambda a, b: a == b + 1,
    "next_to": lambda a, b: abs(a - b) == 1,
    "left_of": lambda a, b: a < b,
    "right_of": lambda a, b: a > b,
    "one_between": lambda a, b: abs(a - b) == 2,
    "two_between": lambda a, b: abs(a - b) == 3,
}


def solve_rules(rules, puzzle):
    """Apply translated rules with a CSP solver.

    Returns (grid_or_None, n_valid, n_invalid). The rules come from model
    output and can have any malformed shape: anything that does not match
    the schema exactly counts as invalid and is skipped. Contradictory rule
    sets return None (infeasible).
    """
    cats, N = puzzle["categories"], puzzle["N"]
    p = Problem()
    for c, vs in cats.items():
        for v in vs:
            p.addVariable(f"{c}::{v}", list(range(N)))
        p.addConstraint(AllDifferentConstraint(), [f"{c}::{v}" for v in vs])

    def valid_pair(x):
        return (isinstance(x, (list, tuple)) and len(x) == 2
                and isinstance(x[0], str) and isinstance(x[1], str)
                and x[0] in cats and x[1] in cats[x[0]])

    n_valid = n_invalid = 0
    for r in rules:
        try:
            t = r.get("type") if isinstance(r, dict) else None
            if t in ("at", "not_at"):
                item, house = r.get("item"), r.get("house")
                if valid_pair(item) and isinstance(house, int) and 1 <= house <= N:
                    pos = house - 1
                    fn = (lambda tp: lambda a: a == tp) if t == "at" else (lambda tp: lambda a: a != tp)
                    p.addConstraint(fn(pos), [f"{item[0]}::{item[1]}"])
                    n_valid += 1
                else:
                    n_invalid += 1
            elif t in _PAIR_CONSTRAINTS:
                a, b = r.get("a"), r.get("b")
                if valid_pair(a) and valid_pair(b) and (a[0] != b[0] or a[1] != b[1]):
                    p.addConstraint(_PAIR_CONSTRAINTS[t], [f"{a[0]}::{a[1]}", f"{b[0]}::{b[1]}"])
                    n_valid += 1
                else:
                    n_invalid += 1
            else:
                n_invalid += 1
        except Exception:
            n_invalid += 1

    try:
        sol = p.getSolution()
    except Exception:
        return None, n_valid, n_invalid
    if not sol:
        return None, n_valid, n_invalid
    grid = {c: [None] * N for c in cats}
    for key, pos in sol.items():
        c, v = key.split("::", 1)
        grid[c][pos] = v
    return grid, n_valid, n_invalid


def grade_grid(grid, puzzle):
    """Fraction of grid cells matching the true solution (0.0 for no grid)."""
    if grid is None:
        return 0.0
    sol, cats, N = puzzle["solution"], list(puzzle["categories"].keys()), puzzle["N"]
    hits = sum(1 for c in cats for i in range(N)
               if grid[c][i] and grid[c][i].lower() == sol[c][i].lower())
    return hits / (len(cats) * N)


# Training dataset: the prompt asks for rules; the reward re-derives the grid
def puzzle_to_row(puzzle):
    return {
        "prompt": [{"role": "user", "content": translation_prompt(puzzle)}],
        "puzzle_solution": json.dumps(puzzle["solution"]),
        "puzzle_categories": json.dumps(puzzle["categories"]),
        "puzzle_n": puzzle["N"],
        "num_clues": len(puzzle["clues"]),
    }

train_dataset = Dataset.from_list([puzzle_to_row(p) for p in train_puzzles])
print(f"Training dataset: {len(train_dataset)} rows")
print(f"Prompt preview:\n{translation_prompt(train_puzzles[0])[:400]}")

---
## 7. Baseline: How Good Is It Before Training?

**What**: Measure the untrained model on both evaluation sets, and cache the
result.

**Why**: Without a baseline, "after training the model scores 0.51" means
nothing. Three numbers are reported, and they answer different questions:

| metric | question it answers |
|---|---|
| `parse_rate` | did the model emit a JSON array at all? |
| `cell_acc` | of the 25 grid cells, what fraction came out right? |
| `puzzle_acc` | how often was the *whole* grid correct? |

`puzzle_acc` is the headline and the harshest: every one of ~15 clues must be
translated correctly, so it behaves roughly like per-clue accuracy raised to the
fifteenth power. That is why small gains in translation quality show up as large
jumps in puzzles solved — and why it starts near the floor.

**Why sampling, not greedy**: generation uses temperature 0.7 / top-p 0.8 /
top-k 20 with thinking disabled, the values Qwen's own model card gives for this
mode. Qwen documents that greedy decoding degrades its output and can send it
into loops, so "temperature 0 for a clean headline number" would be a worse
measurement, not a purer one.

**How**: The result is cached to `eval_baseline_rules.json`. Training changes the
model in place, so once you have trained, the untrained numbers are unrecoverable
without reloading — the cache means a kernel restart does not cost you your
comparison. Delete the file to force a fresh baseline.

In [ ]:
def evaluate_model(model, tokenizer, puzzles, batch_size=16,
                   max_new_tokens=MAX_COMPLETION_LENGTH):
    """Translate clues to rules, solve, grade. Batched generation.

    Returns dict with puzzle_acc (fully solved fraction), cell_acc,
    parse_rate, rewards, samples.
    """
    FastLanguageModel.for_inference(model)
    tok = getattr(tokenizer, "tokenizer", tokenizer)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    rewards, samples = [], []
    parsed = 0

    for start in range(0, len(puzzles), batch_size):
        batch = puzzles[start:start + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": translation_prompt(p)}],
                tokenize=False, add_generation_prompt=True,
                enable_thinking=False,
            )
            for p in batch
        ]
        # Left padding keeps every prompt flush with the generation start
        enc = tok(texts, return_tensors="pt", padding=True,
                  padding_side="left").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **enc, max_new_tokens=max_new_tokens, do_sample=True,
                temperature=EVAL_TEMPERATURE, top_p=EVAL_TOP_P, top_k=EVAL_TOP_K,
                pad_token_id=tok.pad_token_id,
            )
        decoded = tok.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                   skip_special_tokens=True)

        for pz, text in zip(batch, decoded):
            rules = parse_rules(text)
            if rules is None:
                rewards.append(0.0)
                continue
            parsed += 1
            grid, n_valid, n_invalid = solve_rules(rules, pz)
            acc = grade_grid(grid, pz)
            rewards.append(acc)
            if len(samples) < 2:
                samples.append({
                    "clues": pz["clues"][:3],
                    "rules": rules[:3],
                    "n_valid": n_valid, "n_invalid": n_invalid,
                    "acc": acc,
                })

        done = len(rewards)
        if done % 32 == 0 or done == len(puzzles):
            print(f"  {done}/{len(puzzles)}, cell_acc={sum(rewards)/len(rewards):.3f}")

    FastLanguageModel.for_training(model)
    return {
        "puzzle_acc": sum(1 for r in rewards if r == 1.0) / len(rewards),
        "cell_acc": sum(rewards) / len(rewards),
        "parse_rate": parsed / len(puzzles),
        "rewards": rewards,
        "samples": samples,
    }

In [ ]:
# Baseline evaluation is expensive; reuse a cached result — but only one
# measured at this scale, since a baseline over 50 puzzles cannot be compared
# against a trained run over 200. Delete the file to force a fresh baseline.
BASELINE_CACHE = f"eval_baseline_rules_{NUM_EVAL_PUZZLES}x{NUM_BENCH_PUZZLES}.json"

if os.path.exists(BASELINE_CACHE):
    with open(BASELINE_CACHE) as f:
        _cached = json.load(f)
    baseline_indist = _cached["indist"]
    baseline_zlogic = _cached["zlogic"]
    print(f"Loaded cached baseline from {BASELINE_CACHE}")
else:
    print(f"Baseline: in-distribution eval ({NUM_EVAL_PUZZLES} puzzles)...")
    baseline_indist = evaluate_model(model, tokenizer, eval_puzzles)

    print(f"\nBaseline: ZebraLogic 5-house ({NUM_BENCH_PUZZLES} puzzles)...")
    baseline_zlogic = (evaluate_model(model, tokenizer, zlogic_puzzles[:NUM_BENCH_PUZZLES])
                       if zlogic_puzzles else None)

    with open(BASELINE_CACHE, "w") as f:
        json.dump({"indist": baseline_indist, "zlogic": baseline_zlogic}, f)
    print(f"Cached to {BASELINE_CACHE}")

# The comparison in section 11 is only meaningful if both sides measured the
# same puzzles — fail loudly here rather than print a misleading table later.
assert len(baseline_indist["rewards"]) == NUM_EVAL_PUZZLES, (
    f"cached baseline covers {len(baseline_indist['rewards'])} puzzles, "
    f"but this run evaluates {NUM_EVAL_PUZZLES}"
)

print(f"\nBaseline Results:")
print(f"  In-dist    solved={100*baseline_indist['puzzle_acc']:.0f}%  "
      f"cell_acc={baseline_indist['cell_acc']:.3f}  parse={100*baseline_indist['parse_rate']:.0f}%")
if baseline_zlogic:
    print(f"  ZebraLogic solved={100*baseline_zlogic['puzzle_acc']:.0f}%  "
          f"cell_acc={baseline_zlogic['cell_acc']:.3f}  parse={100*baseline_zlogic['parse_rate']:.0f}%")

print("\n--- Sample translation (in-dist) ---")
for s in baseline_indist["samples"][:1]:
    for clue, rule in zip(s["clues"], s["rules"]):
        print(f"  {clue}")
        print(f"    -> {json.dumps(rule)}")
    print(f"  valid={s['n_valid']} invalid={s['n_invalid']} acc={s['acc']:.2f}")

---
## 8. The Reward: Graded by a Solver, Not by a Human

**What**: The function GRPO calls to score every attempt the model makes.

**Why**: This is what makes the whole approach work without a labelled dataset.
For each generated answer the reward function parses the rules, runs the solver,
and compares the deduced grid to the true one:

| outcome | score |
|---|---|
| no JSON array in the output | `0` |
| rules parse and solve | fraction of the 25 cells correct, `0.0 – 1.0` |
| one rule per clue | `+0.05` format bonus |
| grid entirely correct | `+0.25` bonus |

No reward model, no human preferences, no answer key written by anyone — the
grader is a constraint solver and the ground truth came from the generator. That
is the ingredient real tasks usually lack, and the reason puzzles make such a
good gym: **verifiability**.

The two bonuses shape behaviour rather than measure it. The format bonus rewards
answering every clue instead of the easy ones. The perfection bonus keeps a
complete solution clearly above a nearly-complete one, so the last few rules stay
worth fixing.

**How**: TRL passes the extra dataset columns (`puzzle_solution`,
`puzzle_categories`, `puzzle_n`, `num_clues`) straight through to this function.
It runs on CPU, on eight generations per prompt per step — the solver is fast
enough that this is not the bottleneck.

In [ ]:
def zebra_reward_func(completions, puzzle_solution, puzzle_categories,
                      puzzle_n, num_clues, **kwargs):
    """GRPO reward: solver-graded cell accuracy of the translated rules.

    - unparseable output: 0
    - parseable: fraction of correct grid cells (0..1)
    - +0.05 format bonus for one rule per clue
    - +0.25 bonus for a fully correct grid
    """
    rewards = []
    for completion, sol_json, cats_json, n_val, n_clues in zip(
        completions, puzzle_solution, puzzle_categories, puzzle_n, num_clues
    ):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        puzzle = {
            "solution": json.loads(sol_json),
            "categories": json.loads(cats_json),
            "N": int(n_val),
        }
        rules = parse_rules(text)
        if rules is None:
            rewards.append(0.0)
            continue
        grid, n_valid, n_invalid = solve_rules(rules, puzzle)
        score = grade_grid(grid, puzzle)
        if len(rules) == int(n_clues):
            score += 0.05
        if grid is not None and grade_grid(grid, puzzle) == 1.0:
            score += 0.25
        rewards.append(float(score))
    return rewards

print("Reward function ready.")

---
## 9. GRPO Trainer Setup

**What**: Configure the trainer.

**Why — how GRPO learns**: for each prompt the model writes
`NUM_GENERATIONS = 8` different answers. All eight are scored, and the trainer
compares them **against each other**: whatever the above-average attempts did
becomes more likely, whatever the below-average ones did becomes less likely.
There is no target answer to imitate — the group is its own yardstick, which is
where "Group Relative" comes from. It also means the model must already produce
*some* variation in quality for learning to have a foothold; a model that fails
identically every time gives GRPO nothing to compare.

**How**: The rollout sampling values again come from the Qwen3.5 model card
(temperature 1.0, top-p 1.0, top-k 40 for reasoning-style tasks), with thinking
disabled through `chat_template_kwargs`. Unsloth will announce that it has raised
the batch size to match `num_generations` — that is expected.

**A note on speed, and on reading version errors**: Unsloth compiles the
model's hot paths with `torch.compile`, which is where much of its speed
advantage comes from, and that compilation is enabled here. Two failures in this
area look like your bug and are not:

- Training dies at step 0 with `FailOnRecompileLimitHit: Hard failure due to
  fullgraph=True`, and raising `torch._dynamo.config.recompile_limit` changes
  nothing. Under torch 2.12 that config became thread-local, so a limit raised
  on the main thread never reaches the backward worker threads. It means Unsloth
  is older than the PyTorch it runs against; updating Unsloth is the fix.
- A compiled kernel complains about a tensor without storage. That is the
  mismatch described in section 3 — an extension built against a different
  PyTorch than the one loaded.

Both are environment problems wearing the costume of a code problem. When
`torch.compile` is involved, check what your libraries were built against before
you debug your own code.


In [ ]:
training_config = GRPOConfig(
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, MAX_STEPS // 10),
    optim="paged_adamw_8bit",
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    pad_to_multiple_of=64,
    max_steps=MAX_STEPS,
    max_grad_norm=0.1,
    beta=KL_BETA,
    # Rollout sampling per the Qwen3.5 model card (non-thinking reasoning tier)
    temperature=1.0,
    top_p=1.0,
    top_k=40,
    chat_template_kwargs={"enable_thinking": False},
    logging_steps=1,
    save_steps=MAX_STEPS,
    report_to="none",
    output_dir=OUTPUT_DIR,
    seed=SEED,
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[zebra_reward_func],
    args=training_config,
    train_dataset=train_dataset,
)

print(f"Trainer ready: {MAX_STEPS} steps, {NUM_GENERATIONS} generations/prompt")

---
## 10. Training

**What**: Run GRPO and plot the loss.

**Why it takes as long as it does**: each step generates 8 answers for each of 4
prompts, scores all 32 with the solver, and then updates the adapter. Generation
dominates, and it runs uncompiled (section 1). At demo scale expect roughly
**2–2.5 hours for 60 steps** on a single GPU; full scale is an overnight run.

**What to watch**: the loss curve is the least informative thing here — in GRPO
it reflects how far the policy moved, not how good the answers are. The number
that matters is the reward, and the honest way to read it is the before/after
comparison in the next section. Interrupting with Ctrl-C leaves the model in
memory and the adapter usable; the cell catches it deliberately.

**How**: A checkpoint is written at the end of the run. If the kernel dies, the
adapter is recoverable from `OUTPUT_DIR/checkpoint-*` — but the untrained
baseline is not, which is why section 7 cached it.

In [ ]:
start_time = time.time()

try:
    trainer.train()
except KeyboardInterrupt:
    print("\nTraining interrupted — model state is recoverable.")

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed/60:.1f} minutes")

try:
    import matplotlib.pyplot as plt
    losses = [(e["step"], e["loss"]) for e in trainer.state.log_history if "loss" in e]
    if losses:
        steps, vals = zip(*losses)
        plt.figure(figsize=(10, 4))
        plt.plot(steps, vals, alpha=0.7)
        plt.xlabel("Step"); plt.ylabel("Loss")
        plt.title("GRPO Training Loss")
        plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
except ImportError:
    losses = [(e["step"], e["loss"]) for e in trainer.state.log_history if "loss" in e]
    for step, loss in losses[-10:]:
        print(f"  step {step}: {loss:.4f}")

---
## 10b. Record the Run in Thinkube Experiments

**What**: Open a run in **Thinkube Experiments** — the platform's experiment
tracker and model registry, powered by MLflow — log the hyperparameters, and replay the whole
training history into it — every step's reward, loss and completion length.

**Why**: Section 14 will register the trained model, and a model in a registry
raises an obvious question six months later: *how was this made, and was it any
good?* A registered version with no experiment behind it can't answer either.
This is the tracking half of Thinkube Experiments that section 14's table
describes — the half that records what you tried and what happened.

### 💡 Why log after training, not during

TRL can stream metrics live (`report_to="mlflow"` in `GRPOConfig`), and
for short runs that's the neater choice. It is avoided here for one practical
reason: Thinkube Experiments authenticates with a **bearer token that expires in
minutes to an hour**, while this run lasts many hours. A live stream would
start failing partway through, quietly, and you would discover the gap only when
you went looking for the metrics.

`trainer.state.log_history` keeps every logged step in memory regardless, so
replaying it once at the end — with a token obtained seconds earlier — gets the
same data with none of the exposure. The trade is that nothing appears in
Thinkube Experiments until the run ends.

**How**: The run id is kept in `TRAINING_RUN_ID`. Section 11 appends the
evaluation results to it, and section 14 registers the model **against this same
run**, so the model version points back at the experiment that produced it.

In [ ]:
import mlflow
import thinkube_models as tkm

MLFLOW_URI = "http://mlflow.mlflow.svc.cluster.local:5000"
MLFLOW_EXPERIMENT = "zebra-grpo"


def mlflow_connect():
    """Point the MLflow client at the platform, with a fresh bearer token.

    Tokens are short-lived, so this is called again before any later write
    rather than relying on the one obtained here.
    """
    os.environ["MLFLOW_TRACKING_TOKEN"] = tkm.get_mlflow_token() or ""
    mlflow.set_tracking_uri(MLFLOW_URI)
    return mlflow.MlflowClient()


client = mlflow_connect()
exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
EXPERIMENT_ID = exp.experiment_id if exp else mlflow.create_experiment(MLFLOW_EXPERIMENT)

run = client.create_run(
    experiment_id=EXPERIMENT_ID,
    tags={"mlflow.runName": f"grpo-{'full' if FULL_RUN else 'demo'}-{MAX_STEPS}steps"},
)
TRAINING_RUN_ID = run.info.run_id

# Everything that shapes the result, so the run can be reproduced from its record
for k, v in {
    "base_model": MODEL_NAME,
    "scale": "full" if FULL_RUN else "demo",
    "max_steps": MAX_STEPS,
    "train_puzzles": NUM_TRAIN_PUZZLES,
    "eval_puzzles": NUM_EVAL_PUZZLES,
    "bench_puzzles": NUM_BENCH_PUZZLES,
    "train_seed": TRAIN_SEED,
    "eval_seed": EVAL_SEED,
    "seed": SEED,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE,
    "num_generations": NUM_GENERATIONS,
    "grad_accum": GRADIENT_ACCUMULATION,
    "kl_beta": KL_BETA,
    "max_completion_length": MAX_COMPLETION_LENGTH,
    "max_seq_len": MAX_SEQ_LEN,
    "eval_temperature": EVAL_TEMPERATURE,
    "eval_top_p": EVAL_TOP_P,
    "eval_top_k": EVAL_TOP_K,
    "training_minutes": round(elapsed / 60, 1),
}.items():
    client.log_param(TRAINING_RUN_ID, k, v)

# Replay the per-step history. Non-numeric entries (epoch markers, the final
# summary dict) are skipped rather than guessed at.
NUMERIC = (int, float)
logged = 0
for entry in trainer.state.log_history:
    step = entry.get("step")
    if step is None:
        continue
    for key, value in entry.items():
        if key == "step" or not isinstance(value, NUMERIC) or isinstance(value, bool):
            continue
        client.log_metric(TRAINING_RUN_ID, key.replace("/", "_"), float(value), step=int(step))
        logged += 1

client.log_metric(TRAINING_RUN_ID, "baseline_indist_cell_acc", baseline_indist["cell_acc"])
client.log_metric(TRAINING_RUN_ID, "baseline_indist_puzzle_acc", baseline_indist["puzzle_acc"])
if baseline_zlogic:
    client.log_metric(TRAINING_RUN_ID, "baseline_bench_cell_acc", baseline_zlogic["cell_acc"])

print(f"MLflow run: {TRAINING_RUN_ID}")
print(f"  experiment: {MLFLOW_EXPERIMENT} (id {EXPERIMENT_ID})")
print(f"  logged {logged} metric points across {len([e for e in trainer.state.log_history if 'step' in e])} steps")
print(f"  view: https://experiments.{os.environ.get('DOMAIN_NAME','thinkube.com')}/#/experiments/{EXPERIMENT_ID}/runs/{TRAINING_RUN_ID}")

---
## 11. Post-Training Evaluation

**What**: Re-run both evaluations on the trained model and print the comparison.

**Why**: Same puzzles, same sampling settings, same grader as the baseline —
only the adapter changed. Anything else would make the difference unreadable.

What a full run produced on this hardware (250 steps, one GB10):

| | baseline | trained | change |
|---|---|---|---|
| in-dist puzzles solved | 8% | **99%** | +92 points |
| in-dist cell accuracy | 0.313 | **0.995** | +0.682 |
| ZebraLogic puzzles solved | 28% | **67%** | +39 points |
| ZebraLogic cell accuracy | 0.527 | **0.782** | +0.255 |
| parse rate (both sets) | 97–99% | **100%** | — |

Read those rows carefully, because they say different things. **In-distribution
the model is close to saturated** — 99% of puzzles solved on the phrasings it
trained on. That number on its own would be unremarkable: a model can reach it by
learning the generator's habits.

**The row that matters is ZebraLogic**, a public benchmark written by other
people, which the model never saw. Solving 28% before and 67% after means the
skill transferred to unfamiliar phrasing rather than being memorised. Transfer is
the demanding result, and it is the one to quote.

A detail worth noticing: the parse rate was already 97–99% before training. The
model could always produce well-formed rules — what it learned was to produce the
*correct* ones. This task rewards precision in translation, not fluency in
syntax.

A second detail, visible in the training curve: reward reached its ceiling
around step 190 and stayed flat. The last sixty steps bought nothing measurable,
so a shorter run would likely have reached a similar place. When you adapt this
loop, watch for that plateau rather than assuming more steps are better.

**How**: Both evaluations take a few minutes at demo scale. The target check at
the bottom is a guard-rail, not a grade — it tells you whether this run moved
enough to be worth registering.

In [ ]:
print("Trained: in-distribution eval...")
trained_indist = evaluate_model(model, tokenizer, eval_puzzles)

print("\nTrained: ZebraLogic 5-house...")
trained_zlogic = (evaluate_model(model, tokenizer, zlogic_puzzles[:NUM_BENCH_PUZZLES])
                  if zlogic_puzzles else None)

# Comparison table
print(f"\n{'':26s} {'baseline':>10s} {'trained':>10s} {'delta':>10s}")
print(f"{'='*58}")

for label, base, trained in [("in-dist", baseline_indist, trained_indist),
                             ("zlogic", baseline_zlogic, trained_zlogic)]:
    if not (base and trained):
        continue
    for metric, fmt in [("puzzle_acc", "{:9.0%}"), ("cell_acc", "{:10.3f}"),
                        ("parse_rate", "{:9.0%}")]:
        d = trained[metric] - base[metric]
        print(f"{label + '  ' + metric:26s} "
              f"{fmt.format(base[metric]):>10s} {fmt.format(trained[metric]):>10s} "
              f"{('+' if d >= 0 else '') + fmt.format(d).strip():>10s}")

d_solved = trained_indist["puzzle_acc"] - baseline_indist["puzzle_acc"]
d_cell = trained_indist["cell_acc"] - baseline_indist["cell_acc"]
print()
if d_solved >= 0.10 or d_cell >= 0.10:
    print(f"In-dist target MET: solved {d_solved:+.0%} / cell {d_cell:+.3f}")
else:
    print(f"In-dist target NOT met: solved {d_solved:+.0%} / cell {d_cell:+.3f} (want +10% either)")

if baseline_zlogic and trained_zlogic:
    d_z = trained_zlogic["cell_acc"] - baseline_zlogic["cell_acc"]
    print(f"ZebraLogic transfer: cell {d_z:+.3f} "
          f"({'positive transfer' if d_z > 0 else 'no transfer yet'})")

# Append the results to the training run, so the experiment record carries what
# the model achieved and not only how it was trained.
client = mlflow_connect()
for k, v in {
    "trained_indist_cell_acc": trained_indist["cell_acc"],
    "trained_indist_puzzle_acc": trained_indist["puzzle_acc"],
    "trained_indist_parse_rate": trained_indist["parse_rate"],
    "delta_indist_cell_acc": d_cell,
    "delta_indist_puzzle_acc": d_solved,
}.items():
    client.log_metric(TRAINING_RUN_ID, k, float(v))
if baseline_zlogic and trained_zlogic:
    for k, v in {
        "trained_bench_cell_acc": trained_zlogic["cell_acc"],
        "trained_bench_puzzle_acc": trained_zlogic["puzzle_acc"],
        "delta_bench_cell_acc": trained_zlogic["cell_acc"] - baseline_zlogic["cell_acc"],
    }.items():
        client.log_metric(TRAINING_RUN_ID, k, float(v))
print(f"\nResults logged to MLflow run {TRAINING_RUN_ID}")

---
## 12. What the Model Actually Learned

**What**: Take one unseen puzzle, show every clue beside the rule the model wrote
for it, and mark the solver's grid cell by cell against the truth.

**Why**: Aggregate numbers tell you *whether* it improved; this tells you *how*,
and it is where the interesting failures live. Typical things to look for:

- **Direction flips** — `left_of` where the clue said right, or arguments the
  right way round but the wrong rule type. The relation was understood, the
  orientation was not.
- **Invented rule types or fields** — the model reaching for a form the language
  does not have. These are skipped as invalid, and you can see the cost in the
  `invalid` count.
- **One bad rule, whole grid wrong** — a single contradictory rule can push the
  solver to a different arrangement entirely. That is the exponent from section 7
  made visible, and it is why per-clue accuracy is worth chasing.

**How**: Re-run the cell for a different sample; each run re-generates, so
different puzzles expose different weaknesses. This view is also the fastest way
to decide what a longer run should fix.

In [ ]:
from IPython.display import display, Markdown

# One puzzle, clue by clue: what the trained model translated and what the
# solver deduced from it
pz = eval_puzzles[0]
tok = getattr(tokenizer, "tokenizer", tokenizer)
FastLanguageModel.for_inference(model)
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": translation_prompt(pz)}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False)
enc = tok(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=MAX_COMPLETION_LENGTH,
                         do_sample=True, temperature=EVAL_TEMPERATURE,
                         top_p=EVAL_TOP_P, top_k=EVAL_TOP_K,
                         pad_token_id=tok.pad_token_id)
reply = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
FastLanguageModel.for_training(model)

rules = parse_rules(reply) or []
grid, n_valid, n_invalid = solve_rules(rules, pz) if rules else (None, 0, 0)

lines = ["### Clue → rule\n"]
for i, clue in enumerate(pz["clues"]):
    rule = json.dumps(rules[i]) if i < len(rules) else "(missing)"
    lines.append(f"- {clue}\n  - `{rule}`")
lines.append(f"\n**Solver:** valid={n_valid} invalid={n_invalid} "
             f"acc={grade_grid(grid, pz):.2f}")
if grid:
    lines.append("\n| house | " + " | ".join(grid.keys()) + " |")
    lines.append("|" + "---|" * (len(grid) + 1))
    for i in range(pz["N"]):
        marks = []
        for c in grid:
            ok = grid[c][i] and grid[c][i].lower() == pz["solution"][c][i].lower()
            marks.append(f"{grid[c][i]} {'✓' if ok else '✗'}")
        lines.append(f"| {i+1} | " + " | ".join(marks) + " |")
display(Markdown("\n".join(lines)))

---
## 13. Save the Adapter

**What**: Write the LoRA adapter next to the notebook.

**Why**: This is the actual product of the training — roughly 60 MB of adapter
weights, not a 9 GB model. It only means anything alongside the base model it was
trained on, which is why the config records that pairing. Keeping the adapter
separate is what makes fine-tunes cheap to store and to share: one base model on
disk, many adapters beside it.

**How**: Local to this pod. The next section is what turns it into something the
rest of the cluster can use.

In [ ]:
import os

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

abs_path = os.path.abspath(OUTPUT_DIR)
print(f"LoRA adapter saved to: {abs_path}")
print(f"Contents: {os.listdir(abs_path)}")

---
## 14. Close the Loop: Register Your Model in Thinkube Experiments

**What**: Merge the adapter into the base weights and register the result **against
the training run from section 10b**, so the cluster treats your fine-tune like any
other model — with its provenance attached.

### 💡 Platform concept: Thinkube Experiments has two halves

Confusing them is the most common source of "where did my model go?":

| | **Tracking** (experiments & runs) | **Model Registry** |
|---|---|---|
| holds | metrics, parameters, artifacts of one run | named, versioned models |
| answers | "what did I try, and what happened?" | "what is deployable, and which version?" |
| you see it at | `experiments.thinkube.com` → Experiments | → Models |
| identified by | `run_id` | `name` + `version` |

Section 10b wrote the Tracking half. This cell adds the Registry half and, by
passing `run_id=TRAINING_RUN_ID`, **links them**: open the model version in
Thinkube Experiments and you can walk straight back to the reward curve, the
hyperparameters and the evaluation results that produced it. A version registered without that
link is a set of weights nobody can account for.

### 💡 Where the bytes actually go

The artifacts do not live inside Thinkube Experiments. It stores a URI; the files
go to an S3-compatible gateway backed by JuiceFS. That is why a registered model is
readable from **every GPU node** — and why the upload below talks to S3 directly
rather than streaming through the tracking server.

Note the token is refreshed after the upload finishes: a multi-gigabyte transfer
can easily outlive the bearer token that started it.

### 🔁 The alternative: hand it to the platform

```python
# An Argo workflow does the same steps off-notebook — the right choice for a
# long job or a scheduled pipeline:
#   register_finetuned_model(name=…, source_path=…, base_model=…,
#                            server_type="vllm", quantization="BF16")
#   get_mirror_status(workflow_id=…)   # poll until "succeeded"
```

It expects the merged model in the staging area and runs unattended. The
trade-off is that it creates its own run, so the link back to your training
experiment is yours to make.

### 💡 Why merge first

An adapter needs its base model plus a library that knows how to apply it. A
**merged** checkpoint is an ordinary model directory any serving stack can open,
which is what vLLM expects in the next section. The cost is ~9 GB of disk and a
few minutes; the benefit is that your model stops being a special case.

### ⚠️ One more step before it can be served

Registering makes a model *exist*. The serving registry decides what is
*loadable* by reading the **catalog** — so a fine-tune also needs an entry in
`thinkube-metadata/models.json` with `server_type: ["vllm"]` and
`is_finetuned: true`. Register without that and the model shows in Thinkube
Experiments but
`load_model` answers "not found" — which is exactly the error to recognise.

In [ ]:
import base64
import shutil
from pathlib import Path
import boto3
from boto3.s3.transfer import TransferConfig
from botocore.config import Config as BotoConfig
from kubernetes import client as k8s_client, config as k8s_config

FINETUNED_NAME = "zebra-rules-qwen35-4b"
RUN_TAG = f"{'full' if FULL_RUN else 'demo'}-{MAX_STEPS}steps"

# Merge the LoRA into the base weights and write an HF-format checkpoint to the
# MLflow staging area on JuiceFS. A leftover merge from an earlier run would
# silently register the wrong model, so staging is keyed to this run and
# replaced when it does not match.
staging_dir = tkm.STAGING_PATH / FINETUNED_NAME
stamp = staging_dir / ".run_tag"
if staging_dir.exists() and (not stamp.exists() or stamp.read_text().strip() != RUN_TAG):
    print(f"Staging holds a different run; removing {staging_dir}")
    shutil.rmtree(staging_dir)

if not (staging_dir / "config.json").exists():
    model.save_pretrained_merged(str(staging_dir), tokenizer, save_method="merged_16bit")
    stamp.write_text(RUN_TAG)
print(f"Merged model in staging: {staging_dir}  [{RUN_TAG}]")

# The platform's register workflow reads staging from a private volume that
# JupyterHub cannot reach, so the notebook registers directly: it uploads the
# files through the same S3 gateway MLflow uses (credentials read with the
# pod's service account) and creates the registry entry itself.
k8s_config.load_incluster_config()
sec = k8s_client.CoreV1Api().read_namespaced_secret("mlflow-s3-secret", "mlflow")
creds = {k: base64.b64decode(v).decode() for k, v in sec.data.items()}

s3 = boto3.client(
    "s3",
    endpoint_url=creds["S3_ENDPOINT_URL"],
    aws_access_key_id=creds["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=creds["AWS_SECRET_ACCESS_KEY"],
    region_name="us-east-1",
    config=BotoConfig(request_checksum_calculation="when_required",
                      s3={"payload_signing_enabled": False}),
)
transfer = TransferConfig(multipart_threshold=64 * 2**20,
                          multipart_chunksize=64 * 2**20, max_concurrency=4)

# Register against the training run from section 10b — that is what ties the
# deployable model version to the experiment that produced it.
client = mlflow_connect()
prefix = f"artifacts/{EXPERIMENT_ID}/{TRAINING_RUN_ID}/artifacts/model"

files = sorted(p for p in Path(staging_dir).rglob("*")
               if p.is_file() and p.name != ".run_tag")
print(f"Uploading {len(files)} files to s3://mlflow/{prefix}")
for p in files:
    key = f"{prefix}/{p.relative_to(staging_dir)}"
    size_gb = p.stat().st_size / 2**30
    if size_gb > 0.06:
        print(f"  {p.name} ({size_gb:.1f} GB)...")
    s3.upload_file(str(p), "mlflow", key, Config=transfer)
print("Upload complete")

# A long upload can outlive the token that started it
client = mlflow_connect()
try:
    client.create_registered_model(FINETUNED_NAME)
except Exception:
    pass
version = client.create_model_version(
    name=FINETUNED_NAME,
    source=f"s3://mlflow/{prefix}",
    run_id=TRAINING_RUN_ID,
)
print(f"Registered {FINETUNED_NAME} version {version.version}")
print(f"  linked to training run {TRAINING_RUN_ID}")

---
## 14b. Make the Model Loadable

**What**: Two records that turn the registered version into a model the LLM Gateway can load.

**Why**: Registering puts the weights in Thinkube Experiments. The gateway loads a model when it also has:

| record | where | what it says |
|---|---|---|
| a catalogue entry | `models.json` in your private `<your GitHub user>-metadata` repository | how to serve the weights: backend, reasoning format, tool use |
| a completed registration | the `model_mirror_jobs` table of thinkube-control's database | the weights are in place |

The cell writes both. Your catalogue entry wins over a platform entry with the same `id`, and thinkube-control reads the catalogue again within 5 minutes. The gateway checks the registrations every minute, so the cell waits until the model shows `deployable`.

> **Note**: this step is manual in this release. The next releases of Thinkube record a registered fine-tune for you, and this cell goes away.

In [ ]:
import base64
import json
import time
import uuid

import psycopg2
import requests
from tk_llm import LLMClient

CATALOG_ENTRY = {
    "id": FINETUNED_NAME,
    "name": "Zebra Rules (Qwen3.5-4B fine-tune)",
    "params_b": 4.7,
    "active_params_b": None,
    "quantization": "BF16",
    "context_length": 262144,
    "description": "Qwen3.5-4B GRPO-tuned to translate zebra-puzzle clues into formal rules. Produced by the zebra-grpo example notebook.",
    "server_type": ["vllm"],
    "task": "text-generation",
    "reasoning_format": "qwen3",
    "tool_use": False,
    "stop_tokens": [],
    "license": "apache-2.0",
    "gated": False,
    "serving_name": FINETUNED_NAME,
    "is_finetuned": True,
}

# 1. The catalogue entry, in models.json of <user>/<user>-metadata
github = requests.Session()
github.headers.update({
    "Authorization": f"Bearer {os.environ['GITHUB_TOKEN']}",
    "Accept": "application/vnd.github+json",
})
user = github.get("https://api.github.com/user", timeout=30)
user.raise_for_status()
repo = f"{user.json()['login']}/{user.json()['login']}-metadata"
if github.get(f"https://api.github.com/repos/{repo}", timeout=30).status_code == 404:
    raise RuntimeError(f"{repo} does not exist; create it as a private repository on GitHub, then run this cell again")

contents_url = f"https://api.github.com/repos/{repo}/contents/models.json"
current = github.get(contents_url, timeout=30)
if current.status_code == 404:
    catalog, sha = {"models": []}, None
else:
    current.raise_for_status()
    catalog = json.loads(base64.b64decode(current.json()["content"]))
    sha = current.json()["sha"]

others = [m for m in catalog["models"] if m["id"] != FINETUNED_NAME]
if others + [CATALOG_ENTRY] != catalog["models"]:
    body = {
        "message": f"Add {FINETUNED_NAME} to the model catalogue",
        "content": base64.b64encode(
            (json.dumps({**catalog, "models": others + [CATALOG_ENTRY]}, indent=2) + "\n").encode()
        ).decode(),
    }
    if sha:
        body["sha"] = sha
    github.put(contents_url, json=body, timeout=30).raise_for_status()
    print(f"Catalogue entry written to {repo}/models.json")
else:
    print(f"Catalogue entry already in {repo}/models.json")

# 2. The completed registration, in thinkube-control's database
conn = psycopg2.connect(
    host=os.environ["POSTGRES_HOST"],
    port=int(os.environ["POSTGRES_PORT"]),
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname="thinkube_control",
)
with conn, conn.cursor() as cur:
    cur.execute(
        """
        INSERT INTO model_mirror_jobs (id, model_id, status, workflow_name, error_message)
        VALUES (%s, %s, 'succeeded', NULL, NULL)
        ON CONFLICT (model_id) DO UPDATE
        SET status = 'succeeded', error_message = NULL, updated_at = CURRENT_TIMESTAMP
        """,
        (str(uuid.uuid4()), FINETUNED_NAME),
    )
conn.close()
print(f"Registration recorded for {FINETUNED_NAME}")

# 3. Wait until the gateway lists the model as deployable
llm = LLMClient()
deadline = time.time() + 600
while True:
    listed = {m.id: m for m in llm.list_models().models}
    if FINETUNED_NAME in listed:
        print("state:", listed[FINETUNED_NAME].state)
        break
    if time.time() > deadline:
        raise RuntimeError(f"{FINETUNED_NAME} is not listed by the gateway after 10 minutes")
    time.sleep(15)

---
## 15. Serve It Through the LLM Gateway

**What**: Load your fine-tune onto a vLLM backend and call it the way any
application would.

### 💡 Platform concept: the model lifecycle

A registered model is not a running model. It moves through states, and knowing
them turns most "why can't I call it?" moments into a one-line diagnosis:

```
   in catalog ──mirror/register──▶ deployable ──load_model()──▶ loading ──▶ available
                                        ▲                                      │
                                        └──────────── unload_model() ──────────┘
```

| state | meaning | what to do |
|---|---|---|
| *not listed* | not in the catalog | add it to `models.json` |
| `deployable` | weights are on shared storage, no GPU holds it | `load_model()` |
| `loading` | a backend pod is starting and reading weights | wait, 1–5 min |
| `available` | a backend is serving it; the gateway routes to it | call it |

### 💡 Nodes, slots, and why the load needs a target

GPUs are time-sliced into **slots**, so several models can share one card. The
scheduler places a model on a node with a free slot — but it **refuses a node
whose GPU metrics it cannot read**, because placing a model blind risks
exhausting memory for everything already there. That is a safety property, not an
obstacle: the cell below filters for `available_slots > 0` *and*
`metrics_available`, which is the correct way to choose a node.

### 🔁 The reusable surface

Everything you need for your own model, and identical to what notebook 01 used
for stock models:

```python
from tk_llm import LLMClient, get_openai_client

llm = LLMClient()
llm.list_models()                     # catalog + state of each
llm.get_load_options("my-model")      # which backends and nodes could take it
llm.load_model("my-model", node="tkspark")
llm.get_model_status("my-model")      # poll until state == available
llm.unload_model("my-model")          # give the GPU back

client = get_openai_client()          # standard OpenAI client, local endpoint
client.chat.completions.create(model="my-model", messages=[...])
```

**Why a gateway at all**, rather than calling vLLM directly: one endpoint fronts
every backend (vLLM, Ollama, TensorRT-LLM, TEI), so your application names a
model and nothing else. Swap a 4B fine-tune for a 27B stock model by changing a
string. Because the surface is OpenAI-compatible, LangChain, AG2, CrewAI and
LlamaIndex all work unmodified — which is how notebooks 02 and 03 used it.

### 🧩 What the final call demonstrates

The gateway returns **rules**, and the **solver** finishes the job locally. That
is the deployed shape of this whole notebook — the model formalizes, the machine
deduces — and the reason the answer can be trusted: a solver cannot produce a
grid that contradicts the rules it was given. When you apply this pattern to your
own domain, this is the division to keep.

### ⬇️ When you are done

`llm.unload_model(FINETUNED_NAME)` frees the slot. The model stays in Thinkube Experiments and
reloads in minutes, so unload freely — GPU memory is the scarce resource, not
registry space.

In [ ]:
import time
from tk_llm import LLMClient, get_openai_client

llm = LLMClient()

# Pick a GPU node the scheduler can verify (metrics available) with a free slot
opts = llm.get_load_options(FINETUNED_NAME)
node = next(n.name for n in opts.gpu_nodes
            if n.available_slots > 0 and getattr(n, "metrics_available", True))
print(llm.load_model(FINETUNED_NAME, node=node))

# Wait until the backend reports the model as available
for _ in range(60):
    status = llm.get_model_status(FINETUNED_NAME)
    if str(status.model.state) in ("ModelState.available", "available"):
        break
    time.sleep(10)
print("state:", status.model.state, "backend:", status.model.backend_id)

# The served fine-tune, reached through the LLM gateway from any client:
# translate a fresh puzzle's clues, then let the local solver finish the job
client = get_openai_client()
pz = eval_puzzles[1]
reply = client.chat.completions.create(
    model=FINETUNED_NAME,
    messages=[{"role": "user", "content": translation_prompt(pz)}],
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
text = reply.choices[0].message.content
rules = parse_rules(text) or []
grid, n_valid, n_invalid = solve_rules(rules, pz) if rules else (None, 0, 0)
print(f"Gateway translation: {len(rules)} rules, valid={n_valid}, "
      f"acc={grade_grid(grid, pz):.2f}")

---
## [Optional] Export to GGUF

**What**: Convert the merged model to GGUF, the format Ollama and llama.cpp read.

**Why**: vLLM serves this model well on a datacentre GPU. GGUF is the other end
of the spectrum — quantised weights that run on a laptop or a small edge box, at
some cost in quality. If the point of fine-tuning a 4B model was to deploy it
somewhere modest, this is the last step.

**How**: Left commented on purpose. Unsloth's GGUF export is best-effort for
architectures as recent as Qwen3.5, and a failure here should not interrupt a
successful run. Uncomment when you need it, and treat a conversion error as
"not yet supported" rather than as a problem with your training.

In [ ]:
# Uncomment to export:
# model.save_pretrained_gguf("zebra_gguf", tokenizer, quantization_method="q4_k_m")
# print("GGUF export complete.")

---
## 🎓 Summary: You Trained and Deployed Your Own Model

### ✅ What you did

1. ✅ **Loaded a base model from the cluster's mirror** instead of downloading it
2. ✅ **Built a verifiable task** — a generator that produces its own ground truth
   and a rule language a solver can execute
3. ✅ **Trained with GRPO and no labelled data** — the reward was a solver's
   grading of the model's output
4. ✅ **Measured honestly** — cached baseline, two evaluation sets, one of them a
   public benchmark
5. ✅ **Registered the fine-tune in Thinkube Experiments** so every GPU node can read it
6. ✅ **Served it on vLLM and called it through the LLM Gateway** — same client,
   same API as every stock model

### 🎯 Key takeaways — the platform surface

Everything here works for *any* model you train, not just this one:

```python
# ── Getting a model into the cluster ────────────────────────────────
# Catalog entry first (thinkube-metadata/models.json), then:
#   submit_model_mirror(model_id="org/model")     # HF → Experiments/JuiceFS
#   get_mirror_status(workflow_id="model-dl-…")   # running → succeeded

# ── Publishing a model you produced ─────────────────────────────────
model.save_pretrained_merged(path, tokenizer, save_method="merged_16bit")
#   register_finetuned_model(name=…, source_path=…, base_model=…,
#                            server_type="vllm", quantization="BF16")
#   …or register from the notebook with mlflow.start_run() +
#      client.create_model_version(), as section 14 does

# ── Serving and calling it ──────────────────────────────────────────
from tk_llm import LLMClient, get_openai_client
llm = LLMClient()
llm.list_models()                          # catalog + state
llm.get_load_options("my-model")           # nodes with free, measurable slots
llm.load_model("my-model", node="tkspark") # deployable → loading → available
llm.unload_model("my-model")               # give the GPU back

client = get_openai_client()               # OpenAI-compatible, local
client.chat.completions.create(model="my-model", messages=[...])
```

**Three rules of thumb worth keeping**

- **Mirror before you train.** A base model fetched per node is a tax you pay
  forever; mirrored, it is paid once.
- **Register before you serve.** Thinkube Experiments makes a model *exist*; the catalog makes
  it *loadable*. A model missing from `models.json` returns "not found" from
  `load_model` no matter how healthy its Thinkube Experiments entry looks.
- **Unload when you are done.** Registry space is cheap; GPU slots are not.

### 📊 What the training achieved

| | baseline | trained | change |
|---|---|---|---|
| in-dist puzzles solved | 8% | **99%** | +92 points |
| in-dist cell accuracy | 0.313 | **0.995** | +0.682 |
| ZebraLogic puzzles solved | 28% | **67%** | +39 points |
| ZebraLogic cell accuracy | 0.527 | **0.782** | +0.255 |
| rules that parsed | 97–99% | **100%** | — |

250 steps, 414 minutes of training on one GB10, about 9.5 hours for the whole
notebook. ZebraLogic is a public benchmark the model never trained on: it went
from solving about one puzzle in four to two in three.

### 🔍 What that number is, and what it is not

Be precise about what improved, because the honest claim is strong enough
without inflating it.

**What the model got better at**: turning a sentence into the right rule. At
baseline it already emitted well-formed JSON — the parse rate was 99% before
training. What it got wrong was *which* rule and *in which order*: whether "the
Old Gold is somewhere to the right of the red" is a `left_of` with its arguments
swapped, and whether "red" belongs to the colour category or the smoke category.
That mapping is what 250 steps taught it, and the benchmark transfer shows it
learned a mapping rather than memorising phrasings.

**What the model did not get better at**: deduction. It never searches the grid,
never propagates a constraint, never eliminates a candidate. For a 5×5 puzzle
that search space is about 24 billion assignments, and the solver clears it in
milliseconds. Ask this trained model to solve a zebra puzzle *without* the
solver and there is no reason to expect it to do better than before — nothing
here measured that, and nothing here trained it.

So this is not a reasoning model, and the improvement is not reasoning ability.
It is a **large accuracy gain on a narrow, verifiable task**, which is what
fine-tuning is genuinely good for. The reasoning was deliberately moved to a
component that cannot get it wrong. That division is why the numbers are this
good: asking gradient descent to install a search algorithm is a much harder
problem, and one this loop never attempts.

---

## 🔭 Where This Technique Goes Next

The model never solved a puzzle. It turned sentences into exact structure and let
a machine do the rest — and *that* is the skill worth deploying. Every task below
has the property section "Why a Puzzle?" asked for: **a program can tell whether
the output is right**, so the same loop applies with the rule language and the
solver swapped out.

### 🕸️ Knowledge graphs from documents

The closest relative of this notebook. The model emits
`(entity, relation, entity)` triples against your ontology; the verifier checks
that entity types match the relation's signature, that no contradiction is
introduced, and that constraints hold. Training data comes from the graph you
already have: take verified subgraphs, generate or collect the text that
describes them, and you have pairs — exactly the trick the puzzle generator uses.

On thinkube the pieces are already there: **Qdrant** for the embeddings,
**PostgreSQL** or a graph store for the triples, and this notebook's loop for the
extractor. Notebook 02 built retrieval over unstructured text; a graph is what
you build when you need to *query relationships* rather than find passages.

### 🗄️ Questions into queries

The model emits SQL, a Cypher query, or a Qdrant filter. Verification is the
strongest of any task here: **run it**. Correct rows returned is a perfect
reward, execution errors are a clean negative, and partial credit comes from row
overlap. Training data is any question-and-answer pair from your own systems, or
generated by taking existing queries and asking a large model to write the
question they answer.

### 📄 Documents into records

Invoices, contracts, lab reports, clinical notes → structured fields. Verifiers
are unglamorous and effective: schema validation, type checks, totals that must
sum, dates that must fall in range, codes that must exist in a reference table.
Partial credit is per-field, which is a good gradient. This is where a small
fine-tuned model most often beats prompting a large one — the format is narrow,
the volume is high, and the cost per document matters.

### ⚙️ Configuration, commands, and code

The model emits a Kubernetes manifest, a Terraform plan, an API call, a shell
pipeline. Verification is a dry run, a schema, a linter, or a test suite. This is
the same family as the code-RL work behind recent reasoning models, and the
grader is the one your CI already runs.

### 📜 Policies into checks

Written rules — compliance requirements, business logic, eligibility criteria —
turned into executable predicates. Verify by running the predicates against cases
with known outcomes. Slower to set up because you need those cases, but the
payoff is a rule engine that stays in step with the prose it came from.

### 🧪 And the ones to avoid

Be equally clear about where this does *not* work. Summarisation, tone, "is this
a good explanation", open-ended writing — anything whose judge is a person. There
RL needs a reward model trained on human preferences, which is a much larger
project than this notebook. Reach for supervised fine-tuning, or better prompts,
and keep RL for the places where a program can hold the scoresheet.

### 🚀 Next steps here

- **Train longer** — `FULL_RUN = True`. Whole-puzzle accuracy behaves roughly
  like per-clue accuracy to the fifteenth power, so lifting translation from ~90%
  to ~95% per rule would take solved puzzles from 20% to around 45%.
- **Widen the phrasing** — add clue templates to `zebra_dataset.py`. Benchmark
  transfer is limited by how many ways your training data says the same thing.
- **Try the 9B** — set `MODEL_NAME = "unsloth/Qwen3.5-9B"`. It starts stronger and
  costs more to train and serve. Measure rather than assume.
- **Port the loop** — keep sections 6, 8 and 9 (rule language, reward, trainer)
  and replace the puzzle with your own verifiable task. Sections 3, 14 and 15 —
  mirror, register, serve — do not change at all.

---

**Licence**: Apache-2.0. Copyright (c) 2026 Thinkube Contributors.
SPDX-License-Identifier: Apache-2.0

Built with [Unsloth](https://github.com/unslothai/unsloth) (Apache-2.0) and
[TRL](https://github.com/huggingface/trl) (Apache-2.0). Puzzle generation and
scoring by the accompanying `zebra_dataset.py`. The formalize-then-solve approach
follows the neuro-symbolic literature — Logic-LM, SatLM, LINC.

**Benchmark citation:**

```
ZebraLogic: Lin, B.Y., Le Bras, R., Choi, Y. (2024).
ZebraLogic: Benchmarking the Logical Reasoning Ability of Language Models.
https://huggingface.co/datasets/WildEval/ZebraLogic
```